In [7]:
############################################################
#   FULL MULTI-MODEL BENCHMARK PIPELINE (PRINT RESULTS ONLY)
#   Segments: 1s, 2s, 5s, 10s, 20s
#   Models: ML + DL (see below)
############################################################

import os
import time
import numpy as np
import librosa
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

# ML MODELS
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings("ignore")


############################################################
# CONFIG
############################################################
ROOT_DATASET = "/home/feliciano/Documents/DATASET_SEABREAM_SEGMENTED/"

SEGMENT_FOLDERS = {
    "1s":  "Segmented_1s",
    "2s":  "Segmented_2s",
    "5s":  "Segmented_5s",
    "10s": "Segmented_10s",
    "20s": "Segmented_20s"
}

SAMPLE_RATE = 8000
N_MELS = 128
FMAX = 1000
TARGET_T = 200  # resize width → prevents OOM
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BEHAV_CLASSES = ["background", "feeding", "post-feeding", "pre-feeding"]


############################################################
# FEATURE EXTRACTION (ML)
############################################################
def extract_ml_features(path):
    y, sr = librosa.load(path, sr=SAMPLE_RATE)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20).mean(axis=1)
    zcr = librosa.feature.zero_crossing_rate(y).mean()
    sc = librosa.feature.spectral_centroid(y=y, sr=sr).mean()
    bw = librosa.feature.spectral_bandwidth(y=y, sr=sr).mean()
    return np.concatenate([mfcc, [zcr, sc, bw]])


############################################################
# MEL SPECTROGRAM (DL)
############################################################
def extract_mel(path):
    y, sr = librosa.load(path, sr=SAMPLE_RATE)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, fmax=FMAX)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    mel_t = torch.tensor(mel_db).unsqueeze(0).unsqueeze(0)
    mel_r = F.interpolate(mel_t, size=(128, TARGET_T),
                          mode="bilinear", align_corners=False)
    return mel_r.squeeze(0).squeeze(0).float()


############################################################
# LABEL DETECTOR
############################################################
def map_label(path):
    path = path.lower()
    for c in BEHAV_CLASSES:
        if c in path:
            return c
    return None


############################################################
# DATASET CLASS (DL)
############################################################
class MelDataset(Dataset):
    def __init__(self, files, labels):
        self.f = files
        self.y = labels

    def __len__(self):
        return len(self.f)

    def __getitem__(self, idx):
        mel = extract_mel(self.f[idx]).unsqueeze(0)
        lab = torch.tensor(self.y[idx], dtype=torch.long)
        return mel, lab


############################################################
# DL MODELS
############################################################

class MLP(nn.Module):
    def __init__(self, input_size=262, num_classes=4):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return F.log_softmax(self.fc3(x), dim=1)


class CNN1D(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 16, 5)
        self.pool = nn.MaxPool1d(2)
        self.conv2 = nn.Conv1d(16, 32, 5)
        self.fc = nn.Linear(32 * 90, num_classes)

    def forward(self, x):
        x = x.squeeze(1)
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.flatten(1)
        return F.log_softmax(self.fc(x), dim=1)


class GRUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(128, 64, batch_first=True)
        self.fc = nn.Linear(64, 4)

    def forward(self, x):
        x = x.squeeze(1).transpose(1, 2)
        _, h = self.gru(x)
        return F.log_softmax(self.fc(h[-1]), dim=1)


class LSTMNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(128, 64, batch_first=True)
        self.fc = nn.Linear(64, 4)

    def forward(self, x):
        x = x.squeeze(1).transpose(1, 2)
        _, (h, _) = self.lstm(x)
        return F.log_softmax(self.fc(h[-1]), dim=1)


class CNN2D(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d((2, 2))

        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d((2, 2))

        self.gap = nn.AdaptiveAvgPool2d((8, 8))
        self.fc = nn.Linear(64 * 8 * 8, 4)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.gap(x).flatten(1)
        return F.log_softmax(self.fc(x), dim=1)


############################################################
# DL TRAINING FUNCTION (FIXED NAME)
############################################################
def train_dl_model(model, train_loader, test_loader):
    model = model.to(DEVICE)
    opt = optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()

    # Train
    start = time.time()
    for epoch in range(3):
        for mel, label in train_loader:
            mel, label = mel.to(DEVICE), label.to(DEVICE)
            opt.zero_grad()
            out = model(mel)
            loss = crit(out, label)
            loss.backward()
            opt.step()
    train_time = time.time() - start

    # Eval
    y_true, y_pred = [], []
    start_inf = time.time()
    with torch.no_grad():
        for mel, label in test_loader:
            mel = mel.to(DEVICE)
            pred = model(mel).argmax(1).cpu().numpy()
            y_true.extend(label.numpy())
            y_pred.extend(pred)
    inf_time = (time.time() - start_inf) / len(test_loader)

    return y_true, y_pred, train_time, inf_time


############################################################
# RUN EXPERIMENTS
############################################################

ALL_RESULTS = {}

for seg, folder in SEGMENT_FOLDERS.items():

    print(f"\n======== RUNNING {seg} ========")

    # Load files
    path = os.path.join(ROOT_DATASET, folder)

    files, labels = [], []
    for root, dirs, fs in os.walk(path):
        for f in fs:
            if f.endswith(".wav"):
                full = os.path.join(root, f)
                lbl = map_label(root)
                if lbl:
                    files.append(full)
                    labels.append(lbl)

    le = LabelEncoder()
    y = le.fit_transform(labels)

    # ML FEATURES
    X_ml = np.array([extract_ml_features(f) for f in files])
    X_ml = StandardScaler().fit_transform(X_ml)

    X_train_ml, X_test_ml, y_train_ml, y_test_ml = train_test_split(
        X_ml, y, test_size=0.2, stratify=y
    )

    # ML MODELS
    ml_models = {
        "RF": RandomForestClassifier(),
        "ExtraTrees": ExtraTreesClassifier(),
        "SVM": SVC(kernel="rbf"),
        "XGBoost": XGBClassifier(),
        "KNN": KNeighborsClassifier(),
        "Logistic": LogisticRegression(max_iter=500),
        "GaussianNB": GaussianNB(),
        "DecisionTree": DecisionTreeClassifier(),
        "AdaBoost": AdaBoostClassifier()
    }

    seg_results = {}

    # RUN ML MODELS
    for name, model in ml_models.items():
        model.fit(X_train_ml, y_train_ml)
        pred = model.predict(X_test_ml)
        seg_results[name] = f1_score(y_test_ml, pred, average="macro")

    # DL DATASETS
    train_f, test_f, y_train_dl, y_test_dl = train_test_split(
        files, y, test_size=0.2, stratify=y
    )

    train_loader = DataLoader(MelDataset(train_f, y_train_dl),
                              batch_size=16, shuffle=True)
    test_loader = DataLoader(MelDataset(test_f, y_test_dl),
                             batch_size=16, shuffle=False)

    # DL MODELS
    dl_models = {
        "MLP_D": MLP(),
        "CNN1D_D": CNN1D(),
        "GRU_D": GRUNet(),
        "LSTM_D": LSTMNet(),
        "CNN2D_D": CNN2D(),
    }

    # RUN DL MODELS
    for name, model in dl_models.items():
        y_true, y_pred, _, _ = train_dl_model(model, train_loader, test_loader)
        seg_results[name] = f1_score(y_true, y_pred, average="macro")

    ALL_RESULTS[seg] = seg_results


############################################################
# PRINT MASTER RESULT TABLE
############################################################
import pandas as pd

df = pd.DataFrame(ALL_RESULTS)
print("\n===== FINAL MASTER RESULTS =====")
print(df)


======== RUNNING 1s ========


RuntimeError: mat1 and mat2 shapes cannot be multiplied (2048x200 and 262x256)